In [2]:
%reload_ext sql

In [3]:
%sql sqlite:///../database/jobs.db

Connecting to 'sqlite:///../database/jobs.db'

In [4]:
%config SqlMagic.displaylimit = 20

## What are the top 15 most in-demand skills?

In [6]:
%%sql
%%sql
SELECT skills, COUNT(*) as count
FROM skills
GROUP BY skills
ORDER BY count DESC
LIMIT 15;

Running query in 'sqlite:///../database/jobs.db'

skills,count
SQL,50
Python,14
UML,13
Power BI,12
Data analysis,11
Excel,10
BPMN,10
Data analytics,9
BI,9
REST API,7


SQL dominates, appearing in nearly half of all postings (50 of 114) —
the one non-negotiable skill for Data/BI Analyst roles in Poland. Python
and Power BI follow, reflecting the field's two main paths: programming
and BI dashboarding.

## What is the median salary across all analyzed postings?

In [8]:
%%sql
SELECT AVG(mid_salary) AS median_salary
FROM (
    SELECT mid_salary
    FROM jobs
    WHERE mid_salary IS NOT NULL
    ORDER BY mid_salary
    LIMIT 2 - (SELECT COUNT(*) FROM jobs WHERE mid_salary IS NOT NULL) % 2
    OFFSET (SELECT (COUNT(*) - 1) / 2 FROM jobs WHERE mid_salary IS NOT NULL)
);

Running query in 'sqlite:///../database/jobs.db'

median_salary
23100.0


The median salary across Data/BI Analyst roles is **23,100 PLN** gross
per month. This figure blends all experience levels together - the next
query breaks it down by Junior/Mid/Senior for a clearer picture.

## How does salary vary by seniority level?

In [10]:
%%sql
SELECT
    CASE
        WHEN title LIKE '%Junior%' THEN 'Junior'
        WHEN title LIKE '%Senior%' THEN 'Senior'
        WHEN title LIKE '%Mid%' THEN 'Mid'
        ELSE 'Not specified'
    END AS experience_level,
    COUNT(*) AS count,
    ROUND(AVG(mid_salary), 0) AS avg_salary
FROM jobs
WHERE mid_salary IS NOT NULL
GROUP BY experience_level;

Running query in 'sqlite:///../database/jobs.db'

experience_level,count,avg_salary
Mid,2,15540.0
Not specified,76,21962.0
Senior,14,22547.0


Only 16 of 114 postings mention a seniority level (2 Mid, 14 Senior with
salary listed) — none mention "Junior". Senior (22,547 PLN) and
unspecified (21,962 PLN) averages are close, suggesting most
"unspecified" postings are likely Mid/Senior roles. Small sample sizes
here — read as a rough guide only.

## What percentage of postings require English?

In [19]:
%%sql
SELECT ROUND(SUM(has_english) * 100.0 / COUNT(*), 1) AS pct_english
FROM jobs;

Running query in 'sqlite:///../database/jobs.db'

pct_english
9.6


Only 9.6% of postings (11 of 114) explicitly list English as a required
skill. As noted in NOTES.md, this likely understates the real number —
many postings probably mention English requirements only in the full
job description, which isn't accessible due to the site's robots.txt.

## What percentage of postings are remote?

In [24]:
%%sql
SELECT ROUND(SUM(is_remote) * 100.0 / COUNT(*), 1) AS pct_remote
FROM jobs;

Running query in 'sqlite:///../database/jobs.db'

pct_remote
37.7


37.7% of postings offer remote work — a substantial share, showing
remote options are common for Data/BI Analyst roles in Poland. The
remaining 62.3% are on-site or require a specific location.

## Which cities have the most on-site postings?

In [34]:
%%sql
SELECT location, COUNT(location) AS location_count
FROM jobs
WHERE location NOT IN ("Zdalnie")
GROUP BY location
ORDER BY location_count DESC
LIMIT 15;

Running query in 'sqlite:///../database/jobs.db'

location,location_count
Kraków,27
Warszawa,25
Poznań,6
Wrocław,4
Gdańsk,4
Łódź,3
Gdynia,2


Kraków (27) and Warszawa (25) dominate on-site postings, together
accounting for over half of all location-specific offers. Smaller tech
hubs like Poznań, Wrocław, and Gdańsk follow well behind, each with
single digits.

## Which companies post the most Data/BI Analyst offers?

In [36]:
%%sql
SELECT company, COUNT(company) as company_count
FROM jobs
GROUP BY company
ORDER BY company_count DESC
LIMIT 15;

Running query in 'sqlite:///../database/jobs.db'

company,company_count
Scalo,17
Ework Group,10
Link Group,9
Mindbox Sp. z o.o.,8
Antal,7
Verita HR,5
Xebia sp. z o.o.,4
T-Mobile Polska,4
TP Poland Sp. z o.o.,3
Devire,3


Scalo (17), Ework Group (10), and Link Group (9) post the most offers —
but note these are recruitment agencies/software houses hiring on
behalf of clients, not single companies building large in-house analyst
teams. Among direct employers, Mindbox (8) and T-Mobile Polska (4)
stand out.

## Bonus: Which skills correlate with higher salaries?

In [43]:
%%sql
SELECT skills.skills, ROUND(AVG(jobs.mid_salary), 1) AS avg_salary, COUNT(*) AS count
FROM skills
JOIN jobs ON skills.link = jobs.link
WHERE jobs.mid_salary IS NOT NULL
GROUP BY skills.skills
HAVING COUNT(*) >= 5
ORDER BY avg_salary DESC
LIMIT 15;


Running query in 'sqlite:///../database/jobs.db'

skills,avg_salary,count
Data visualization,27000.0,8
Dashboarding,27000.0,8
Tableau,25948.0,5
Excel,25790.9,11
BI,24780.0,8
Python,24696.0,22
Data modeling,23488.0,5
Spark,23240.0,12
UML,23220.7,14
BPMN,23135.8,12


SQL is the most common skill (67 postings) but its salary is only
mid-range (22,449 PLN). Visualization and BI tools — Data
Visualization, Dashboarding, Tableau (25,000-27,000 PLN) — show up in
the highest-paying postings, even though they're used less often. This
suggests SQL is just the baseline. Being able to visualize and present
data may be what separates the higher-paying roles. Only skills with
at least 5 postings are included, for reliability.